In [1]:
import os
os.environ["OLLAMA_HOST"] = "localhost:11434" 
print(os.environ.get("OLLAMA_HOST"))

localhost:11434


In [2]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'd:\OVGU _Saurabh\SEM 3\Review Project\Code_17Apr\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [4]:
import sys
sys.path.append(r"D:\OVGU _Saurabh\SEM 3\Review Project\Code_17Apr\curren_files")

In [5]:
import Prompts.ver_1 as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_1 import Config
import evaluate
importlib.reload(evaluate)

<module 'evaluate' from 'D:\\OVGU _Saurabh\\SEM 3\\Review Project\\Code_17Apr\\curren_files\\evaluate.py'>

In [6]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [7]:
def evaluate_model(df, results):
    # Get true labels from the dataframe
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [9]:
def run_prompt_eval(dataset_path, prompt_template, model, text_col="text"):
    
    df_local = pd.read_csv(dataset_path)
    reviews = df_local[text_col].tolist()

    prompt = PromptTemplate(
        input_variables=["text"],
        template=prompt_template,
    )

    llm = OllamaLLM(model=model)
    chain = prompt | llm

    predictions = []
    for review in reviews:
        response = chain.invoke({"text": review})
        label = response.strip().lower()

        if "fake" in label:
            cleaned_label = "fake"
        elif "deceptive" in label:
            cleaned_label = "fake"
        elif "real" in label:
            cleaned_label = "real"
        elif "truthful" in label:
            cleaned_label = "real"
        else:
            cleaned_label = "fake"  # safety fallback
        
        cleaned_label = cleaned_label.strip('.,!?;:')

        #print(f"Raw output: {response} -> Cleaned: {cleaned_label}")
        predictions.append(cleaned_label)

    accuracy, f1, conf_matrix = evaluate_model(df_local, predictions)
    return df_local, predictions, accuracy, f1, conf_matrix

# 1. Amazon Human vs Computer

In [19]:
# Load CSV
df_reviews = pd.read_csv("D:\OVGU _Saurabh\SEM 3\Review Project\Code_17Apr\Datasets\Amazon_Human_VS_ComputerFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1


## 1.1 Zero Shot Prompting

In [8]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="D:\OVGU _Saurabh\SEM 3\Review Project\Code_17Apr\Datasets\Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="qwen3.5:9b"
 )

In [9]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.6725

F1 Score: 0.6717594071624101

Confusion Matrix:
[[288 112]
 [150 250]]


## 1.2 One-Shot Prompting

In [15]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="D:\OVGU _Saurabh\SEM 3\Review Project\Code_17Apr\Datasets\Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="qwen3.5:9b"
 )

In [16]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.705

F1 Score: 0.7010463378176383

Confusion Matrix:
[[236 164]
 [ 72 328]]


## 1.3 Few-Shot Prompting

In [17]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="D:\OVGU _Saurabh\SEM 3\Review Project\Code_17Apr\Datasets\Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="qwen3.5:9b"
 )

In [18]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.6975

F1 Score: 0.6965821610371372

Confusion Matrix:
[[257 143]
 [ 99 301]]


# 2. Hotel Human Real vs Human Fake

In [19]:
from Prompts.ver_1 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and deceptive reviews.

    Task: Classify the following hotel or product review as either "truthful" or "deceptive".

    Review: "{text}"

    Classify this review as either "truthful" or "deceptive" (respond with only one word):


In [20]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,truthful,conrad,Hotel,We stayed for a one night getaway with family ...,0,TripAdvisor
1,truthful,hyatt,Hotel,Triple A rate with upgrade to view room was le...,0,TripAdvisor
2,truthful,hyatt,Hotel,This comes a little late as I'm finally catchi...,0,TripAdvisor
3,truthful,omni,Hotel,The Omni Chicago really delivers on all fronts...,0,TripAdvisor
4,truthful,hyatt,Hotel,I asked for a high floor away from the elevato...,0,TripAdvisor


## 2.1 Zero Shot Prompting

In [16]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [17]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5690954773869347

F1 Score: 0.5403685092127303

Confusion Matrix:
[[254 542]
 [144 652]]


## 2.2 One-Shot Prompting

In [21]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b")

In [19]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5483668341708543

F1 Score: 0.49694271250158767

Confusion Matrix:
[[182 614]
 [105 691]]


## 2.3 Few-Shot Prompting

In [22]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
        dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
            prompt_template=Config.few_shot_prompt_template,
                model="llama3:8b"
)


In [23]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5515075376884422

F1 Score: 0.4795384615384615

Confusion Matrix:
[[143 653]
 [ 61 735]]


# 3. Human Real vs CG

In [24]:
from Prompts.ver_1 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and deceptive reviews.

    Task: Classify the following hotel or product review as either "truthful" or "deceptive".

    Review: "{text}"

    Classify this review as either "truthful" or "deceptive" (respond with only one word):


In [25]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web
1,real,palmer,Hotel,Here are my experiences: - Had three rooms res...,0,Web
2,real,omni,Hotel,My wife and I travelled to Chicago and really ...,0,TripAdvisor
3,real,knickerbocker,Hotel,The location of this hotel is excellent. The h...,0,Web
4,real,fairmont,Hotel,We recently completed our second stay at the F...,0,TripAdvisor


## 3.1 Zero Shot Prompting

In [24]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [25]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.4491206030150754

F1 Score: 0.367135737386878

Confusion Matrix:
[[ 71 725]
 [152 644]]


## 3.2 One-Shot Prompting

In [26]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [27]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.48304020100502515

F1 Score: 0.3923604943607828

Confusion Matrix:
[[ 77 719]
 [104 692]]


## 3.3 Few-Shot Prompting

In [28]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [29]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.4748743718592965

F1 Score: 0.3426517335187794

Confusion Matrix:
[[ 21 775]
 [ 61 735]]


# 4. Human Real vs MixFake

In [20]:
from Prompts.ver_1 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and deceptive reviews.

    Task: Classify the following hotel or product review as either "truthful" or "deceptive".

    Review: "{text}"

    Classify this review as either "truthful" or "deceptive" (respond with only one word):


In [24]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,sofitel,Hotel,Just got back from a 10 day visit to Chicago. ...,0,TripAdvisor
1,real,palmer,Hotel,Just returned from a week in Chicago with the ...,0,TripAdvisor
2,real,hardrock,Hotel,Not worth the price. We almost stayed at a hot...,0,Web
3,real,homewood,Hotel,WARNING: FOOD POISONING ALERT!!! The room was ...,0,Web
4,real,ambassador,Hotel,Upon check in - the lobby was BUSY! I was tire...,0,TripAdvisor


## 4.1 Zero Shot Prompting

In [25]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [26]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5201005025125628

F1 Score: 0.47638432555672655

Confusion Matrix:
[[184 612]
 [152 644]]


## 4.2 One-Shot Prompting

In [27]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [28]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5364321608040201

F1 Score: 0.48295310519645124

Confusion Matrix:
[[171 625]
 [113 683]]


## 4.3 Few-Shot Prompting

In [29]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [30]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5094221105527639

F1 Score: 0.4087202155206236

Confusion Matrix:
[[ 77 719]
 [ 62 734]]
